# Week 2 — Collections & Control Flow: Dictionaries & Conditionals
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Create and manipulate dictionaries
- Use if/elif/else for business logic
- Combine loops with conditionals
- Nest dictionaries for structured data

## Why This Matters: One Order, Many Details

Every Olist order carries at least six pieces of information: an ID, a status, a price, a freight cost, a state, and a review score. Yesterday we stored groups of *similar* values in lists — all 8 statuses together, all 8 counts together. But an *order* is different: it is a collection of *different* things that belong together.

A dictionary handles this naturally — it maps descriptive keys (`"status"`, `"price"`, `"state"`) to their values, so you retrieve order details by name rather than by guessing a position. The Olist dataset also shows aggregate data at the state level: SP has 41,746 orders and R$ 5,202,955 in revenue. A nested dict stores this compactly, with state codes as outer keys and a mini-dict of metrics as each value.

Once data is in a dict, **conditionals** let you write business rules against it: is this order a fast delivery? A premium-priced item? A problematic status? `if/elif/else` is how those rules become code — and combining them with loops lets you classify every row at once.

In [ ]:
# Real Olist values — no data loading needed this week
# All figures verified from the Olist dataset

statuses = ["delivered", "shipped", "canceled", "unavailable",
            "invoiced", "processing", "created", "approved"]
counts   = [96478, 1107, 625, 609, 314, 301, 5, 2]

# A single Olist order (real structure, anonymised order_id)
order = {
    "order_id":     "e481f51cbdc54678b7cc49136f2d6af7",
    "status":       "delivered",
    "price":        58.90,
    "freight":      13.29,
    "state":        "SP",
    "review_score": 4
}

print("Olist data ready!")  # expected: Olist data ready!

## §3a — Dictionaries: Looking Up Values by Name

A dictionary is Python's key-value store. Think of it like a physical dictionary: you look up a word (the key) to find its definition (the value). Unlike a list — where you find items by position (`order[0]`, `order[3]`) — a dict lets you look things up by a meaningful name: `order["status"]` is immediately readable; `order[3]` is a mystery.

Dictionaries are **mutable** (you can add, change, or delete entries) and fast at lookup regardless of size. In data analysis, a dictionary almost always represents *one row of data* — a single order, a single customer — where each field name is a key and its value is the dict's value. The three core operations are: **access** with `dict["key"]` (raises `KeyError` if absent), **safe access** with `dict.get("key", default)` (returns a default instead of crashing), and **iteration** with `for key, value in dict.items()`.

In [ ]:
# --- Access values ---
print(order["status"])                       # expected: delivered
print(order["price"])                        # expected: 58.9
print(order.get("review_score"))             # expected: 4
print(order.get("seller", "unknown"))        # expected: unknown  (key absent → default)

# --- Add and update ---
order["total"] = order["price"] + order["freight"]
print(order["total"])                        # expected: 72.19

order["review_score"] = 5                    # update existing key
print(order["review_score"])                 # expected: 5

# --- Iterate all key-value pairs ---
print()
for key, value in order.items():
    print(f"  {key}: {value}")

# --- Keys and values as lists ---
print()
print(list(order.keys()))
print(list(order.values()))

## §3b — Nested Dictionaries: Structured Data in Multiple Levels

A dict's values can themselves be dicts — this is called a **nested dictionary**. Think of it like a filing cabinet: the outer dict is the drawer labelled by state code ("SP", "RJ"), and each drawer contains its own labelled folders ("orders", "revenue"). You navigate to a specific piece of data by chaining two key lookups: `state_summary["SP"]["revenue"]`.

Nested dicts are the natural way to represent tabular data before you have pandas. Each outer key is like a row identifier; the inner dict holds that row's columns. When you do eventually load a CSV into a pandas DataFrame in Week 4, you will see that each column lookup (`df["revenue"]`) is conceptually identical to the inner key lookup you are practising now.

The main rule: every inner dict for the same outer key should have **the same set of inner keys**. If SP has "orders" and "revenue", every other state should too — otherwise your loop code will need defensive `.get()` calls everywhere.

In [ ]:
# --- Dict of dicts: state-level summary (verified Olist data) ---
state_summary = {
    "SP": {"orders": 41746, "revenue": 5202955.05},
    "RJ": {"orders": 12852, "revenue": 1824092.67},
    "MG": {"orders": 11635, "revenue": 1585308.03}
}

# Two-level lookup
print(state_summary["SP"]["orders"])      # expected: 41746
print(state_summary["SP"]["revenue"])     # expected: 5202955.05
print(state_summary["RJ"]["orders"])      # expected: 12852

# Loop the nested structure
print()
for state, data in state_summary.items():
    rev_m = data["revenue"] / 1_000_000
    print(f"  {state}: {data['orders']:,} orders  |  R${rev_m:.1f}M revenue")

# expected:
#   SP: 41,746 orders  |  R$5.2M revenue
#   RJ: 12,852 orders  |  R$1.8M revenue
#   MG: 11,635 orders  |  R$1.6M revenue

# Add a derived metric to each inner dict
for state in state_summary:
    avg = state_summary[state]["revenue"] / state_summary[state]["orders"]
    state_summary[state]["avg_order_value"] = round(avg, 2)

print()
print(state_summary["SP"]["avg_order_value"])   # expected: 124.6

## §3c — Conditionals: Writing Business Rules in Code

An `if/elif/else` block is Python's decision tree: the `if` tests the first condition, each `elif` adds an alternative branch, and `else` is the fallback for everything that did not match. Python evaluates branches top to bottom and executes only the first one that is `True` — so the ordering of conditions matters.

In data analysis, conditionals almost always classify a numerical or categorical value into a label: fast vs slow delivery, premium vs economy pricing, active vs problem status. You write the classification once as a function, then apply it across all rows in a loop (or in a pandas `.apply()` call later). The key design rule: put the most restrictive condition first. For delivery speed, check `<= 7` before `<= 14` — otherwise a 5-day delivery would match `<= 14` first and be labelled "Normal" instead of "Fast".

In [ ]:
# --- Delivery speed classifier ---
# Verified stats: Min=0 days, Median=10 days, Mean=12.1 days, Max=209 days

def classify_delivery(days):
    if days is None:
        return "Unknown"
    elif days <= 7:
        return "Fast"
    elif days <= 14:
        return "Normal"
    elif days <= 30:
        return "Slow"
    else:
        return "Very Slow"

for days in [0, 5, 10, 20, 50, None]:
    print(f"{str(days):>6} days  →  {classify_delivery(days)}")

# expected:
#      0 days  →  Fast
#      5 days  →  Fast
#     10 days  →  Normal
#     20 days  →  Slow
#     50 days  →  Very Slow
#   None days  →  Unknown

print()

# --- Order value classifier ---
# Verified: Avg price = R$120.65, Max = R$6,735.00
def classify_order_value(price):
    if price >= 500:
        return "Premium"
    elif price >= 100:
        return "Standard"
    else:
        return "Economy"

for p in [58.90, 120.65, 500.00, 6735.00]:
    print(f"R${p:>8.2f}  →  {classify_order_value(p)}")

# expected:
#   R$   58.90  →  Economy
#   R$  120.65  →  Standard
#   R$  500.00  →  Premium
#   R$ 6735.00  →  Premium

## §3d — Combining Loops with Conditionals

Classifying one value is useful; classifying every value in a dataset is the real goal. The pattern is simple: loop over a list (or dict), apply a conditional to each item, and accumulate results. You can count how many items fall into each category, build a new dict with the classifications, or filter to only the items that match a condition.

This loop-plus-conditional pattern is the foundation of nearly every data summarisation task. In Week 4 you will replace the manual loop with a pandas one-liner — but the logic is identical. Practising it now means that when you see `df["category"] = df["price"].apply(classify_order_value)` later, you will understand exactly what is happening under the hood: pandas is running the same if/elif/else function on every row.

In [ ]:
# --- Count orders by price category ---
sample_prices = [58.90, 120.65, 45.00, 780.00, 99.99,
                 350.00, 25.00, 6735.00, 115.00, 499.99]

# Initialise counters
economy  = 0
standard = 0
premium  = 0

for price in sample_prices:
    category = classify_order_value(price)
    if category == "Economy":
        economy += 1
    elif category == "Standard":
        standard += 1
    else:
        premium += 1

print(f"Economy:  {economy}")   # expected: Economy:  4
print(f"Standard: {standard}")  # expected: Standard: 4
print(f"Premium:  {premium}")   # expected: Premium:  2

# --- Build a results dict from the loop ---
results = {"Economy": economy, "Standard": standard, "Premium": premium}
print(results)   # expected: {'Economy': 4, 'Standard': 4, 'Premium': 2}

# --- Combine with a loop over a dict to print a report ---
total = sum(results.values())
for category, count in results.items():
    print(f"  {category}: {count} orders ({count/total*100:.0f}%)")

## §4 — Going Deeper: Dict Comprehensions

Just as list comprehensions build a new list in one line, **dict comprehensions** build a new dict in one line. The pattern is `{key: value  for item in iterable  if condition}`. The `if` part is optional.

The most common data-work use is turning two parallel lists into a lookup dict: `{s: c for s, c in zip(statuses, counts)}` gives you `{"delivered": 96478, "shipped": 1107, ...}` in one expression. The group exercise asks you to build exactly this dict; a comprehension is the cleanest way to do it. You can also transform values on the fly — computing percentages, rounding numbers, or filtering to only the rows that meet a threshold — all within the same one-liner.

In [ ]:
# --- Build status → count lookup dict from two parallel lists ---
# Loop version (verbose but transparent):
status_counts_loop = {}
for s, c in zip(statuses, counts):
    status_counts_loop[s] = c
print(status_counts_loop["delivered"])   # expected: 96478

# Dict comprehension (same result, one line):
status_counts = {s: c for s, c in zip(statuses, counts)}
print(status_counts["delivered"])        # expected: 96478
print(status_counts["canceled"])         # expected: 625

# --- Filter: only statuses with more than 500 orders ---
major = {s: c for s, c in zip(statuses, counts) if c > 500}
print(list(major.keys()))
# expected: ['delivered', 'shipped', 'canceled', 'unavailable']

# --- Transform: percentage of total ---
total = sum(counts)
pct_dict = {s: round(c / total * 100, 1) for s, c in zip(statuses, counts)}
print(pct_dict["delivered"])   # expected: 97.0
print(pct_dict["shipped"])     # expected: 1.1

## §5 — Common Mistakes to Avoid

Three mistakes are especially common with dictionaries and conditionals.

**Mistake 1 — KeyError on missing key**: `dict["key"]` raises `KeyError` if the key does not exist in the dict. In real data you will constantly encounter rows with missing fields; always use `dict.get("key", default)` when the key might be absent.

**Mistake 2 — Stacking `if` instead of `elif`**: Using multiple `if` statements instead of `if/elif` means every condition is evaluated independently. A price of R$700 would be counted as both "Standard" (matches `>= 100`) and "Premium" (matches `>= 500`) if you wrote two separate `if` blocks — causing double-counting that is very hard to debug. Always chain related, mutually exclusive conditions with `elif`.

**Mistake 3 — Forgetting `return` in a classification function**: Writing `classify_delivery(days)` without a `return` statement makes the function return `None` silently. Every branch of your classification function needs its own `return`.

In [ ]:
# ── Mistake 1: KeyError vs .get() ────────────────────────────────────────────
sample_order = {"status": "delivered", "price": 120.65}

# WRONG (crashes if key missing):
# print(sample_order["seller"])   # KeyError: 'seller'

# CORRECT — use .get() with a default value:
print(sample_order.get("seller", "unknown"))   # expected: unknown
print(sample_order.get("price", 0.0))          # expected: 120.65

# ── Mistake 2: if/if vs if/elif (double-counting) ────────────────────────────
price = 700.00

# WRONG — two independent if blocks both fire for R$700:
wrong_standard = 0
wrong_premium  = 0
if price >= 100:
    wrong_standard += 1   # fires
if price >= 500:
    wrong_premium  += 1   # also fires — R$700 counted in both!
print(f"Wrong: standard={wrong_standard}, premium={wrong_premium}")
# expected: Wrong: standard=1, premium=1  ← R$700 counted twice

# CORRECT — elif means only one branch fires:
right_standard = 0
right_premium  = 0
if price >= 500:
    right_premium  += 1
elif price >= 100:
    right_standard += 1
print(f"Right: standard={right_standard}, premium={right_premium}")
# expected: Right: standard=0, premium=1  ← R$700 correctly Premium only

# ── Mistake 3: missing return ─────────────────────────────────────────────────
def broken_classifier(price):
    if price >= 500:
        return "Premium"
    elif price >= 100:
        return "Standard"
    # Missing: else: return "Economy"

result = broken_classifier(50.00)
print(result)           # expected: None  ← silent failure, no crash
print(result is None)   # expected: True

## §6 — Mini-Challenge ⏱ ~8 minutes

Using the `state_summary` dict of dicts, classify each state by order volume and print a formatted summary.

**Classification rules:**
- Orders > 15,000 → `"High Volume"`
- Orders 5,000–15,000 → `"Medium Volume"`
- Orders < 5,000 → `"Low Volume"`

**Expected output:**
```
SP: 41,746 orders → High Volume
RJ: 12,852 orders → Medium Volume
MG: 11,635 orders → Medium Volume
```

Then: how many states are "Medium Volume"? Expected: `2`

In [ ]:
# Pre-defined data — do not change these values
state_summary = {
    "SP": {"orders": 41746, "revenue": 5202955.05},
    "RJ": {"orders": 12852, "revenue": 1824092.67},
    "MG": {"orders": 11635, "revenue": 1585308.03}
}

# Task 1: Write a classify_volume(n) function using if/elif/else
# Your code here

# Task 2: Loop through state_summary and print each state with its classification
# Expected: "SP: 41,746 orders → High Volume"
# Your code here

# Task 3: Count how many states are "Medium Volume"
# Expected: 2
# Your code here

## §7 — Group Exercise (40 minutes): Build a Status Summary Dict

Olist records 99,441 orders across 8 status categories. Your job is to build a structured summary dictionary from two parallel lists — ending with a nested lookup table that includes both raw counts and percentages.

Work in your groups. Each task builds directly on the previous one.

**Tasks:**
1. Create a dict mapping `status → count` using `zip()`. Expected: `{"delivered": 96478, "shipped": 1107, ...}`
2. Using a loop, print only statuses where count > 500. Expected: `delivered (96478)`, `shipped (1107)`, `canceled (625)`, `unavailable (609)`
3. Write a function `classify_status(status)` that returns `"Active"` for delivered/shipped/invoiced/processing, `"Problem"` for canceled/unavailable, and `"Other"` otherwise. Test with all 8 statuses.
4. Using a dict comprehension, build a nested dict: `{"delivered": {"count": 96478, "pct": 97.0}, ...}`

In [ ]:
# Verified Olist data — do not change these values
statuses = ["delivered", "shipped", "canceled", "unavailable",
            "invoiced", "processing", "created", "approved"]
counts   = [96478, 1107, 625, 609, 314, 301, 5, 2]

# Task 1: Dict mapping status → count using zip()
# Expected: {"delivered": 96478, "shipped": 1107, ...}
# Your code here

# Task 2: Loop — print only statuses with count > 500
# Expected: delivered (96478), shipped (1107), canceled (625), unavailable (609)
# Your code here

# Task 3: classify_status() function — test with all 8 statuses
# Your code here

# Task 4: Nested dict comprehension — {status: {"count": N, "pct": X.X}, ...}
# Hint: total = sum(counts)
# Your code here

## §8 — Session Summary

| Concept | Key Syntax | Example |
|---|---|---|
| Create a dict | `{key: value, ...}` | `order = {"status": "delivered", "price": 58.90}` |
| Access | `dict["key"]` | `order["status"]` → `"delivered"` |
| Safe access | `dict.get("key", default)` | `order.get("seller", "unknown")` → `"unknown"` |
| Add/update | `dict["key"] = value` | `order["total"] = 72.19` |
| Iterate | `for k, v in dict.items():` | `for state, data in state_summary.items():` |
| Nested dict | `dict[k1][k2]` | `state_summary["SP"]["revenue"]` → `5202955.05` |
| Dict comprehension | `{k: v for k, v in zip(...)}` | `{s: c for s, c in zip(statuses, counts)}` |
| Branch logic | `if … elif … else …` | `classify_delivery(days)` |
| Classify in loop | `for x in data: if/elif` | count Economy/Standard/Premium |

---
**Coming up Week 3**: Functions & First Contact with Data — you will write reusable functions with parameters and return values, then use Python's `csv` module to load a real Olist CSV file row by row. Week 3 is the final step before pandas: by its end you will be loading data with just two lines of code.